# LLM Trainer Masterclass — Orientation & Resource Accounting

**Goal:** learn to think like an LLM trainer before touching a training framework.

By the end of this notebook you should be able to trace a model-development program from **mission → data → model → compute → training → evaluation → deployment → economics**.

> This notebook is deliberately not a fine-tuning recipe. It is the decision layer that prevents expensive engineering mistakes.

## 1. The complete lifecycle

```text
MISSION
  ↓
TASKS + USERS + CONSTRAINTS
  ↓
DATA ───────────────→ EVALUATION SET
  ↓                         ↑
TOKENIZER                  │
  ↓                         │
ARCHITECTURE                │
  ↓                         │
PRETRAIN / ADAPT ───────────┘
  ↓
POST-TRAINING
  ↓
SAFETY + QUALITY
  ↓
INFERENCE / SERVING
  ↓
MONITORING → NEW DATA → NEXT RUN
```

The most common beginner mistake is to start at **“Which model should we fine-tune?”**. A trainer starts at **“What measurable capability must exist, and what is the cheapest reliable way to obtain it?”**

## 2. The intervention ladder

Use the least expensive intervention that satisfies the requirement, then escalate when evidence says the current layer is insufficient.

| Intervention | Changes weights? | Good for | Bad fit |
|---|---:|---|---|
| Prompting | No | behavior, task framing | persistent learned behavior |
| RAG | Usually no | changing/fresh knowledge | deep behavior change |
| Tools/agents | No base-weight change | actions, live systems | pure language style |
| SFT | Yes | instruction following, formatting, task behavior | rapidly changing facts |
| LoRA/QLoRA | Small adapter only | efficient specialization | rebuilding foundational knowledge |
| Continued pretraining | Yes | domain/language adaptation | tiny datasets |
| Full fine-tuning | Yes, all parameters | substantial adaptation of an existing model | large models on small hardware |
| Preference optimization | Yes | ranking/style/preferences | replacing factual knowledge |
| Pretraining from scratch | Entire model | new foundation model, sovereign data/control | ordinary application problems |

### Critical thinking #1

**Question:** A ministry wants a chatbot that answers questions about regulations that change every month. Should we train the model on the regulations?

<details><summary>Answer</summary>

Usually not as the first design. The knowledge is dynamic, so the primary mechanism should be retrieval from authoritative sources, with evaluation for retrieval correctness. Fine-tuning may later be justified for behavior, terminology, or structured responses, but retraining is a poor mechanism for monthly factual updates.
</details>

**Counterexample:** if the requirement is not merely to know the regulations but to reliably follow a specialized reasoning/formatting procedure that existing models cannot learn through prompting, SFT can become useful.

## 3. Resource accounting starts with parameters × tokens

For dense autoregressive Transformer pretraining, a useful first-order estimate is:

\[
C pprox 6ND
\]

where:

- `N` = number of trainable parameters
- `D` = number of training tokens
- `C` = approximate training FLOPs

The estimate is accounting, not a guarantee. Real systems also pay for attention details, embeddings, optimizer operations, communication, input pipelines, evaluation, checkpointing, and inefficiency.

Chinchilla's empirical result is a separate statement: under its experimental setting and fixed-compute objective, model size and training tokens should scale together; the familiar ~20 tokens/parameter ratio is a planning heuristic, not a law of nature.

In [ ]:
def training_flops(params, tokens):
    """First-order dense LM pretraining estimate."""
    return 6 * params * tokens

examples = [
    (100e6, 2e9),      # 100M model, 2B tokens
    (1e9, 20e9),       # 1B model, 20B tokens
    (7e9, 140e9),     # 7B model, 140B tokens
]
for p, d in examples:
    print(f"N={p/1e9:.1f}B, D={d/1e9:.1f}B -> {training_flops(p,d):.2e} FLOPs")

## 4. Turn compute into wall-clock and money

Let:

\[
T pprox rac{C}{G 	imes P_{eff}}
\]

where `G` is the number of GPUs and `P_eff` is **effective** FLOP/s per GPU, not the marketing peak. Efficiency depends on kernels, sequence length, utilization, communication, batch size, precision, and parallelism.

Then:

\[
Cost pprox T_{hours} 	imes G 	imes price_{GPU-hour}
\]

This is intentionally simple. Later notebooks will replace the assumptions with measured throughput from an actual run.

In [ ]:
def estimate_run(params, tokens, gpus, effective_tflops, gpu_hour_price):
    flops = training_flops(params, tokens)
    effective_flops_per_s = effective_tflops * 1e12
    seconds = flops / (gpus * effective_flops_per_s)
    hours = seconds / 3600
    cost = hours * gpus * gpu_hour_price
    return {
        "training_flops": flops,
        "wall_clock_hours": hours,
        "wall_clock_days": hours / 24,
        "gpu_hours": hours * gpus,
        "estimated_cost_usd": cost,
    }

estimate_run(
    params=1e9,
    tokens=20e9,
    gpus=1,
    effective_tflops=100,
    gpu_hour_price=1.0,
)

## Critical thinking #2

**Question:** If a 1B model and a 100B model both have the same training FLOPs, does that mean they will have the same quality?

<details><summary>Answer</summary>

No. Compute is only the budget. How it is allocated between model capacity, tokens, data quality, architecture, optimization, and training stages changes the result. Chinchilla showed this clearly: under a fixed-compute objective, changing the parameter/token allocation materially changed the resulting model quality.
</details>

## 5. The national-LLM lens

For a language or country-specific program, the important inventory is not simply “how many GPUs do we have?”.

| Asset | Question |
|---|---|
| Language data | How much text exists, in what domains, with what rights? |
| Tokenizer | Does the vocabulary represent the target language efficiently? |
| Human expertise | Who defines quality, safety, terminology, and cultural coverage? |
| Evaluation | Can we measure performance before and after training? |
| Compute | What model/data combination fits the available budget? |
| Systems | Can we store data, stream batches, checkpoint, and recover from failures? |
| Deployment | Where will inference run and at what scale? |
| Governance | Who owns the data, model, weights, and evaluation assets? |
| Sustainability | How will the model be updated and paid for? |

## 6. Free-first laboratory strategy

Google Colab provides free access to compute resources including GPUs/TPUs, but the free tier has dynamic, non-guaranteed availability and changing limits. Therefore the course treats Colab as an **instructional laboratory**, not as production infrastructure.

The practical strategy is:

**small model → measure → reproduce → reason about scaling → build the proposal for larger hardware**

We will never teach students to confuse “I ran a 50M model in Colab” with “I know how to operate a 50B training cluster.”

## 7. Video spine

| Video | Use it for | What to do while watching |
|---|---|---|
| [3Blue1Brown — Attention](https://www.youtube.com/watch?v=eMlx5fFNoYc) | visual intuition | stop before each new mechanism and predict the tensor operation |
| [Karpathy — Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY) | code-level foundations | reproduce each section, then change one design choice |
| [Karpathy — Let's reproduce GPT-2](https://www.youtube.com/watch?v=l8pRSuU81PU) | real training systems | track every optimization and ask what it changes in throughput/memory |
| [Raschka — Building LLMs from the Ground Up](https://www.youtube.com/watch?v=quh7z1q7-uc) | end-to-end synthesis | implement the pipeline, then replace the dataset/model |
| [Stanford CS336](https://cs336.stanford.edu/) | academic backbone | use lectures/assignments as deeper exercises |

## 8. Your first engineering exercise

A team says:

> “We have 1,000 GPU-hours and want the largest possible national-language model.”

Do **not** answer with a parameter count yet.

Write down:

1. target language(s);
2. target capabilities;
3. available licensed/public data;
4. desired context length;
5. training objective;
6. baseline model(s);
7. evaluation suite;
8. GPU type and memory;
9. effective throughput assumption;
10. acceptable failure/retry budget.

Then calculate three candidate plans and compare **measured outcome per dollar**, not parameter count alone.

## Sources

- Stanford CS336: https://cs336.stanford.edu/
- Chinchilla: https://arxiv.org/abs/2203.15556
- Llama 3: https://arxiv.org/abs/2407.21783
- OLMo 2: https://allenai.org/olmo2
- Dolma: https://arxiv.org/abs/2402.00159
- FineWeb: https://arxiv.org/abs/2406.17557
- BLOOM: https://www.jmlr.org/papers/v25/23-0581.html
- LoRA: https://arxiv.org/abs/2106.09685
- QLoRA: https://arxiv.org/abs/2305.14314
- InstructGPT: https://arxiv.org/abs/2203.02155
- DPO: https://arxiv.org/abs/2305.18290
- FlashAttention: https://arxiv.org/abs/2205.14135